# Lab 3 Exercise: browser memory and a block list

This notebook covers both tasks at the end of Lab 3.

- Exercise 1 adds a checkpointer and `thread_id`, then checks that the agent remembers an earlier result.
- Exercise 2 adds middleware that blocks listed sites.

Both tasks share the Playwright MCP browser tools.

## Shared setup

Load the browser tools once and use them for both tasks. They are async, so use `await` and `ainvoke`.

In [ ]:
import sys
from urllib.parse import urlparse
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(override=True)

Use the lab's Windows fix for the MCP error log. It does nothing on macOS and Linux.

In [ ]:
if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

In [ ]:
client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools")

## Exercise 1: browser memory

Add `MemorySaver` and reuse one `thread_id` so the agent keeps the conversation.

The follow-up refers to the three stories without repeating them and tells the agent not to reopen the site. A correct answer shows that memory worked.

In [ ]:
memory_browser_agent = create_agent(
    model="openai:gpt-5.5",
    tools=browser_tools,
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "hn-browsing"}}

first = await memory_browser_agent.ainvoke(
    {"messages": [{"role": "user",
        "content": "Go to https://news.ycombinator.com and tell me the titles of the top three stories on the front page."}]},
    config=config,
)
print(first["messages"][-1].content)

In [ ]:
second = await memory_browser_agent.ainvoke(
    {"messages": [{"role": "user",
        "content": "Of those three stories, which one sounds the most technical, and why? Do not open the site again."}]},
    config=config,
)
print(second["messages"][-1].content)

## Exercise 2: navigation block list

The middleware checks each navigation call. For a blocked host, it returns a `ToolMessage` without calling the handler. Other calls continue as normal.

The browser tools are async, so the middleware awaits the handler. Only `browser_navigate` needs a host check.

In [ ]:
BLOCKED_SITES = {"facebook.com", "x.com"}

@wrap_tool_call
async def block_navigation(request, handler):
    call = request.tool_call
    if call["name"] == "browser_navigate":
        url = call["args"].get("url", "")
        host = (urlparse(url).hostname or "").lower().rstrip(".")
        if any(host == site or host.endswith(f".{site}") for site in BLOCKED_SITES):
            print(f"  [blocklist] refused navigation to {url}")
            return ToolMessage(
                content=f"Navigation to {url} was refused: this site is on the block list.",
                tool_call_id=call["id"],
            )
    return await handler(request)

Ask the guarded agent to open a blocked site. The printed block-list message proves that the handler did not run.

In [ ]:
guarded_agent = create_agent(
    model="openai:gpt-5.5",
    tools=browser_tools,
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
    middleware=[block_navigation],
)

blocked = await guarded_agent.ainvoke({"messages": [{"role": "user",
    "content": "Go to https://www.facebook.com and tell me what you see."}]})
print(blocked["messages"][-1].content)

Now open an allowed site to show that other navigation still works.

In [ ]:
allowed = await guarded_agent.ainvoke({"messages": [{"role": "user",
    "content": "Go to https://example.com and tell me the main heading on the page."}]})
print(allowed["messages"][-1].content)